In [1]:
%reload_ext autoreload
%autoreload 2

import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)



In [2]:
from datetime import timedelta

from analyse.advertising.e00_download.mediatree import (
    CachedMediatreeAPI,
    all_intervals_between,
)
from analyse.advertising.e02_finger_printer.finger_printer import (
    FingerprintPipeline,
    print_report,
    plot_groups,
)

from datetime import datetime
from zoneinfo import ZoneInfo
import json
from pathlib import Path

tz_paris = ZoneInfo("Europe/Paris")
channel = "tf1"
output_prefix = "FINGERPRINT_FULLWEEK"

MONDAY_LAST_YEAR = datetime(2025, 3, 10, tzinfo=tz_paris)

SIMILARITY_THRESHOLD = 0.2

OUTPUT_FILE = f"working_data/{output_prefix}_report_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.json"
OUTPUT_PLOT = f"working_data/{output_prefix}_plot_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.png"


In [3]:
api = CachedMediatreeAPI()

INPUT_SEGMENTS_FOLDER = os.path.join(
    "../working_data", f"segments_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}"
)

input_files = []

# CUSTOM_DATES = [
#     (
#         datetime(2025, 3, 10, 12, 0, tzinfo=tz_paris),
#         datetime(2025, 3, 10, 12, 30, tzinfo=tz_paris),
#     ),
#     (
#         datetime(2025, 3, 10, 12, 30, tzinfo=tz_paris),
#         datetime(2025, 3, 10, 13, 0, tzinfo=tz_paris),
#     ),
#     (
#         datetime(2025, 3, 11, 12, 0, tzinfo=tz_paris),
#         datetime(2025, 3, 11, 12, 30, tzinfo=tz_paris),
#     ),
#     (
#         datetime(2025, 3, 11, 12, 30, tzinfo=tz_paris),
#         datetime(2025, 3, 11, 13, 0, tzinfo=tz_paris),
#     ),
# ]

CUSTOM_DATES = all_intervals_between(
    start_date=MONDAY_LAST_YEAR,
    end_date=MONDAY_LAST_YEAR + timedelta(days=7),
    interval=timedelta(minutes=30),
)

for start_date, end_date in CUSTOM_DATES:
    audio_file = await api.download_export(
        channel, start_date, end_date + timedelta(minutes=1)
    )  # on ajoute une minute pour être sûr de couvrir toute la période

    segment_file = os.path.join(
        INPUT_SEGMENTS_FOLDER,
        f"{start_date.strftime('%Y-%m-%d_%H-%M-%S')}.json",
    )
    input_files.append((audio_file, segment_file, start_date.timestamp()))

input_files


[('../exports/tf1_2025-03-09_23-00-00Z_2025-03-09_23-31-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_00-00-00.json',
  1741561200.0),
 ('../exports/tf1_2025-03-09_23-30-00Z_2025-03-10_00-01-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_00-30-00.json',
  1741563000.0),
 ('../exports/tf1_2025-03-10_00-00-00Z_2025-03-10_00-31-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_01-00-00.json',
  1741564800.0),
 ('../exports/tf1_2025-03-10_00-30-00Z_2025-03-10_01-01-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_01-30-00.json',
  1741566600.0),
 ('../exports/tf1_2025-03-10_01-00-00Z_2025-03-10_01-31-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_02-00-00.json',
  1741568400.0),
 ('../exports/tf1_2025-03-10_01-30-00Z_2025-03-10_02-01-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_02-30-00.json',
  1741570200.0),
 ('../exports/tf1_2025-03-10_02-00-00Z_2025-03-10_02-31-00Z.mp3',
  '../working_data/segments_

In [4]:
pipeline = FingerprintPipeline(
    sources=input_files,
    similarity_threshold=0.05,
    min_matching_hashes=1,
    n_peaks_by_segment=5,
    neighborhood_peaks_filter=15,
    min_peak_amplitude=0.01,
)

if False:
    fingerprints = pipeline.fingerprints()
    pipeline.diagnose(fingerprints)
else:
    report, fingerprints, groups = pipeline.run()
    print_report(report)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"  Rapport JSON : {Path(OUTPUT_FILE).absolute()}")


  [Cache] Chargement depuis tf1_2025-03-09_23-00-00Z_2025-03-09_23-31-00Z_4bc9c0cb_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-09_23-30-00Z_2025-03-10_00-01-00Z_a978b110_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_00-00-00Z_2025-03-10_00-31-00Z_57f53e67_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_00-30-00Z_2025-03-10_01-01-00Z_421618c4_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_01-00-00Z_2025-03-10_01-31-00Z_d3208300_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_01-30-00Z_2025-03-10_02-01-00Z_a004726f_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_02-00-00Z_2025-03-10_02-31-00Z_3b42d562_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_02-30-00Z_2025-03-10_03-01-00Z_60f4e3e1_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_03-00-00Z_2025-03-10_03-31-00Z_4ec06846_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_03-30-00Z_2025-03-10_04-01-00Z_12c2f45f_2a1da438.json
  [Cache] Chargement depuis tf

  [Cache] Chargement depuis tf1_2025-03-10_10-00-00Z_2025-03-10_10-31-00Z_37e6c1b6_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_10-30-00Z_2025-03-10_11-01-00Z_d87328d9_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_11-00-00Z_2025-03-10_11-31-00Z_cc104e11_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_11-30-00Z_2025-03-10_12-01-00Z_5890416d_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_12-00-00Z_2025-03-10_12-31-00Z_02e4c995_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_12-30-00Z_2025-03-10_13-01-00Z_0791c229_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_13-00-00Z_2025-03-10_13-31-00Z_51b755c4_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_13-30-00Z_2025-03-10_14-01-00Z_b82cea13_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_14-00-00Z_2025-03-10_14-31-00Z_8339ecec_2a1da438.json
  [Cache] Chargement depuis tf1_2025-03-10_14-30-00Z_2025-03-10_15-01-00Z_408ebddf_2a1da438.json
  [Cache] Chargement depuis tf

Comparaison fingerprints: 100%|██████████| 44294/44294 [00:00<00:00, 86425.06it/s]


  44294 paires similaires trouvées

══════════════════════════════════════════════════════════════════════
  RAPPORT DE RÉPÉTITIONS
══════════════════════════════════════════════════════════════════════
  Segments analysés : 90224
  Groupes détectés  : 76008

  Classification             Groupes   Occurrences tot
  ────────────────────────────────────────────────────
  SEGMENT_UNIQUE               70227             70227
  CONTENU_REPETE                4950             11564
  JINGLE_RARE                    511              5491
  PUBLICITE_RARE                 238              1455
  PUBLICITE                       62              1054
  JINGLE                          11               424
  EMISSION_UNIQUE                  9                 9

  TOP 15 GROUPES LES PLUS FRÉQUENTS :
  Groupe    Occurrences   Durée moy  Classification
  ────────────────────────────────────────────────────────────
  G45928             51       2.9s  JINGLE
    ↳ [2025-03-10 10:31:16 UTC]  2.9s
    ↳ [202

In [ ]:
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)
from analyse.advertising.tools.testimony_table.extract import get_testimony_data
from analyse.advertising.e00_download.mediatree import CachedMediatreeAPI
from analyse.advertising.tools.basic_elements.segments import (
    load_segment_groups_from_json,
)
from analyse.advertising.e01_rupture_detection.rupture_detector import (
    RuptureDetector,
    print_summary,
)
import httpx

mediatree_api = CachedMediatreeAPI()

TESTIMONY_CHANNEL = "TF1"

detector = RuptureDetector(
    sensitivity=0.1,
    context_sec=1.0,  # durée analysée de chaque côté d'un point pour évaluer la rupture.
    max_ruptures=0,  # 0 pour pas de limite
    sr=22050,  # Fréquence d'échantillonnage de travail
    hop_length=512,  # Pas entre frames (~23ms à 22050Hz)
    n_mfcc=10,  # Nombre de coefficients MFCC
    novelty_smooth_sec=0.5,  # Lissage temporel de la courbe de nouveauté.
    min_segment_sec=1.0,  # Durée minimale d'un segment pour être considéré comme tel.
    silence_percentile=8.0,  # Percentile pour définir le seuil de silence
    cosine_weight=1.0,
)

for start_date, end_date in CUSTOM_DATES:
    annotations = get_testimony_data(
        channel=TESTIMONY_CHANNEL,
        from_date=start_date,
        to_date=end_date,
        source_file="export.csv",
    )

    async with httpx.AsyncClient(timeout=httpx.Timeout(60.0, connect=60.0)) as client:
        witness_file = await mediatree_api.api.get_single_export_url(
            client, channel, start_date, end_date, media_format="mp4"
        )

    segment_file = os.path.join(
        INPUT_SEGMENTS_FOLDER,
        f"{start_date.strftime('%Y-%m-%d_%H-%M-%S')}.json",
    )
    segments = load_segment_groups_from_json(segment_file)

    output_path = f"working_data/{output_prefix}/result_{channel}_{start_date.strftime('%Y%m%d_%H%M%S')}.html"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    generate_player(
        media_input=witness_file,
        segments=[s.to_dict() for s in segments],
        annotations=format_annotation(annotations, from_date=start_date),
        output_path=output_path,
        start_epoch=int(start_date.timestamp()),
        params=detector.get_params(),
        # novelty_peaks=detector.get_novelty_peaks(novelty),
        groups=report["groups"],
    )

[1/2] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-09T23-00-00Z_2025-03-09T23-30-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/2] Segments fournis en entrée
  210 segments

Génération du rapport HTML : working_data/FINGERPRINT_FULLWEEK/result_tf1_20250310_000000.html...

✓  Rapport généré  : /root/Workspace/quotaclimat/analyse/advertising/e02_finger_printer/working_data/FINGERPRINT_FULLWEEK/result_tf1_20250310_000000.html
   Taille HTML     : 27.5 Mo (léger, média chargé à la volée)
   URL média       : https://vod.mediatree.fr/assets/tf1_2025-03-09T23-00-00Z_2025-03-09T23-30-00Z.mp4
   Segments        : 210

   → Ouvrez dans Firefox ou Chrome pour l'analyse.

[1/2] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-09T23-30-00Z_2025-03-10T00-00-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/2] Segments fournis en entrée
  221 segments

Génération du rapport HTML : working_data/FINGERPRINT_